# 第 5 章 · Prompt Caching（几乎整文件可执行拷贝）

对照：`../hermes_src/agent/prompt_caching.py`（本文件很短，**几乎全文拷进下一格**）

```mermaid
flowchart LR
    A[apply_anthropic_cache_control] --> B[断点1: system]
    A --> C[断点2~4: 最后可承载的非 system]
```

In [ ]:
# =============================================================================
# SOURCE COPY: hermes_src/agent/prompt_caching.py  （几乎全文）
# =============================================================================
"""Anthropic prompt caching strategy.

Single layout: ``system_and_3``. 4 cache_control breakpoints — system
prompt + last 3 non-system messages, all at the same TTL (5m or 1h).
"""

import copy
from typing import Any, Dict, List


def _apply_cache_marker(msg: dict, cache_marker: dict, native_anthropic: bool = False) -> None:
    """Add cache_control to a single message, handling all format variations."""
    role = msg.get("role", "")
    content = msg.get("content")

    if role == "tool" and native_anthropic:
        msg["cache_control"] = cache_marker
        return

    if content is None or content == "":
        if role == "tool" and not native_anthropic:
            return
        if role == "assistant" and not native_anthropic:
            return
        msg["cache_control"] = cache_marker
        return

    if isinstance(content, str):
        msg["content"] = [
            {"type": "text", "text": content, "cache_control": cache_marker}
        ]
        return

    if isinstance(content, list) and content:
        last = content[-1]
        if isinstance(last, dict):
            last["cache_control"] = cache_marker


def _can_carry_marker(msg: dict, native_anthropic: bool) -> bool:
    if native_anthropic:
        return True
    content = msg.get("content")
    if content is None or content == "":
        return False
    if isinstance(content, list):
        return bool(content) and isinstance(content[-1], dict)
    return isinstance(content, str)


def _build_marker(ttl: str) -> Dict[str, str]:
    marker: Dict[str, str] = {"type": "ephemeral"}
    if ttl == "1h":
        marker["ttl"] = "1h"
    return marker


def apply_anthropic_cache_control(
    api_messages: List[Dict[str, Any]],
    cache_ttl: str = "5m",
    native_anthropic: bool = False,
) -> List[Dict[str, Any]]:
    """Apply system_and_3 caching strategy.

    Places up to 4 cache_control breakpoints: system prompt + last 3 non-system
    messages, all at the same TTL.
    """
    messages = copy.deepcopy(api_messages)
    if not messages:
        return messages

    marker = _build_marker(cache_ttl)
    breakpoints_used = 0

    if messages[0].get("role") == "system":
        _apply_cache_marker(messages[0], marker, native_anthropic=native_anthropic)
        breakpoints_used += 1

    remaining = 4 - breakpoints_used
    non_sys = [
        i
        for i in range(len(messages))
        if messages[i].get("role") != "system"
        and _can_carry_marker(messages[i], native_anthropic=native_anthropic)
    ]
    for idx in non_sys[-remaining:]:
        _apply_cache_marker(messages[idx], marker, native_anthropic=native_anthropic)

    return messages


print("apply_anthropic_cache_control 已定义（源码副本）")

In [ ]:
msgs = [
    {"role": "system", "content": "## MEMORY\n- Docker\n\nYou are Hermes."},
    {"role": "user", "content": "hi"},
    {"role": "assistant", "content": "hello"},
    {"role": "user", "content": "继续"},
    {"role": "assistant", "content": "好的"},
    {"role": "user", "content": "下一步"},
]
out = apply_anthropic_cache_control(msgs)

def has_marker(msg):
    c = msg.get("content")
    if isinstance(c, list):
        return any(isinstance(p, dict) and "cache_control" in p for p in c)
    return "cache_control" in msg

for i, m in enumerate(out):
    print(i, m["role"], "cache_control=", has_marker(m))

# 讲解：中途改 system 字符串 → 前缀失效（与 MemoryStore snapshot 规则呼应）
import hashlib
sp = msgs[0]["content"]
print("\n改 SP 前", hashlib.sha256(sp.encode()).hexdigest()[:16])
print("改 SP 后", hashlib.sha256((sp + "\nEXTRA").encode()).hexdigest()[:16])